# Notebook 7: Advanced Models
In this notebook, we will train two advanced ensemble classifiers on our vectorized features:
1. **Random Forest Classifier** (bagging algorithm)
2. **XGBoost Classifier** (gradient boosting algorithm)

We will evaluate their performance and compare them against our baseline models.

In [1]:
import pandas as pd
import os
import joblib
from src.train import prepare_data, evaluate_predictions
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Load processed dataset
data_path = os.path.join("..", "data", "processed_jobs.csv")
df = pd.read_csv(data_path)

# Prepare features and split
X_train, X_test, y_train, y_test, vectorizer = prepare_data(df)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (14304, 5003)
Test shape: (3576, 5003)


## 1. Train and Evaluate Random Forest Classifier
Random Forest builds multiple decision trees and merges them together to get a more accurate and stable prediction. We balance the class weights.

In [2]:
# Train Random Forest (using max_depth to control training speed on high-dimensional text data)
rf = RandomForestClassifier(n_estimators=100, max_depth=20, class_weight='balanced', random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:, 1]

rf_metrics = evaluate_predictions(y_test, y_pred_rf, y_prob_rf)
print("=== Random Forest Metrics ===")
print("Accuracy:", rf_metrics['accuracy'])
print("Precision (Fake):", rf_metrics['precision'])
print("Recall (Fake):", rf_metrics['recall'])
print("F1 Score:", rf_metrics['f1_score'])
print("ROC-AUC:", rf_metrics['roc_auc'])
print("Confusion Matrix:\n", rf_metrics['confusion_matrix'])

=== Random Forest Metrics ===
Accuracy: 0.9653243847874721
Precision (Fake): 0.6060606060606061
Recall (Fake): 0.8092485549132948
F1 Score: 0.693069306930693
ROC-AUC: 0.9842607423915315
Confusion Matrix:
 [[3312, 91], [33, 140]]


## 2. Train and Evaluate XGBoost Classifier
XGBoost implements gradient boosted decision trees designed for speed and performance. We balance the dataset by scaling the weight of positive class (fraudulent) relative to negative class (legitimate) using the `scale_pos_weight` parameter.

In [3]:
# Calculate scale_pos_weight: ratio of negative to positive cases (17014 / 866 = 19.6)
scale_weight = (len(y_train) - sum(y_train)) / sum(y_train)

xgb = XGBClassifier(n_estimators=100, max_depth=6, scale_pos_weight=scale_weight, random_state=42, n_jobs=-1)
xgb.fit(X_train, y_train)
y_pred_xgb = xgb.predict(X_test)
y_prob_xgb = xgb.predict_proba(X_test)[:, 1]

xgb_metrics = evaluate_predictions(y_test, y_pred_xgb, y_prob_xgb)
print("=== XGBoost Metrics ===")
print("Accuracy:", xgb_metrics['accuracy'])
print("Precision (Fake):", xgb_metrics['precision'])
print("Recall (Fake):", xgb_metrics['recall'])
print("F1 Score:", xgb_metrics['f1_score'])
print("ROC-AUC:", xgb_metrics['roc_auc'])
print("Confusion Matrix:\n", xgb_metrics['confusion_matrix'])

=== XGBoost Metrics ===
Accuracy: 0.9818232662192393
Precision (Fake): 0.8461538461538461
Recall (Fake): 0.7630057803468208
F1 Score: 0.8024316109422492
ROC-AUC: 0.9779928964412563
Confusion Matrix:
 [[3379, 24], [41, 132]]


## 3. Saving the Best Advanced Model
Let's save the XGBoost model to our `models/` directory for reference, alongside the linear model.

In [4]:
xgb_path = os.path.join("..", "models", "xgboost_model.pkl")
joblib.dump(xgb, xgb_path)
print("Successfully saved XGBoost model to:", xgb_path)

Successfully saved XGBoost model to: ..\models\xgboost_model.pkl
